# Tập 1.3 · PID: robot tự sửa sai 🤖

[![Mở trên Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhpro82/tu-hoc-robot-tu-hanh/blob/main/tap13/tap13_pid.ipynb)

Series **Tự Học Robot Tự Hành** · Mùa 1 · ⏱ ~30 phút · Giáo trình: Principles of Robot Autonomy, [Chương 3](https://porabook.com/book/closed-loop-control/)

Robot xuất phát lệch vạch **30 cm**, một bánh yếu luôn kéo nó quay lệch. Trong notebook này bạn:

1. Chạy lại đúng mô phỏng đã tạo ra số liệu trong video.
2. **Tự viết** bộ điều khiển PID (3 bài: P → D → I), có ô chấm tự động.
3. Vẽ đường đi của robot bằng chính PID của bạn, rồi tự chỉnh hệ số.

Chạy lần lượt từng ô từ trên xuống (Shift + Enter).

## 0. Chuẩn bị (chạy 1 lần)

In [ ]:
import os, sys
if "google.colab" in sys.modules:  # trên Colab: tải code về
    if not os.path.exists("/content/tu-hoc-robot-tu-hanh"):
        !git clone -q https://github.com/thanhpro82/tu-hoc-robot-tu-hanh.git /content/tu-hoc-robot-tu-hanh
    os.chdir("/content/tu-hoc-robot-tu-hanh/tap13")
sys.path[:0] = ["bai_tap", "sim", ".."]

import pid as goc                  # mô phỏng + lời giải dùng trong video
from ve_do_thi import ve           # vẽ đường đi
from IPython.display import Image, display
from kiem_tra import cham          # bộ chấm
print("Sẵn sàng!")

## 1. Chạy lại số liệu trong video

In [ ]:
# 4 cách điều khiển: vòng hở, chỉ P, PD, PID. "Lệch cuối" = độ lệch trung bình trong 3 giây cuối.
for ten, path in goc.KET_QUA.items():
    ys = [p[1] for p in path]
    print(f"{ten:7s} -> lệch cuối {goc.lech_cuoi(path):+6.1f} cm | y nhỏ nhất {min(ys) * 100:+6.1f} cm | x cuối {path[-1][0]:.1f} m")

## 2. Bài tập

Bộ điều khiển nhận sai số `e = r − y` (mục tiêu trừ vị trí đo được; vạch là `r = 0`) và trả về lệnh quay:

$$\omega = K_p\,e + K_i\int e\,dt + K_d\,\frac{de}{dt}$$

| Bài | Hàm | Việc |
|---|---|---|
| **1** | `thanh_phan_P(e)` | trả về `Kp·e` |
| **2** | `thanh_phan_D(e)` | `de/dt ≈ (e − e_trước)/dt`; bước đầu chưa có `e_trước` thì coi `de/dt = 0`; nhớ lưu `e` vào `self.e_truoc` |
| **3** | `thanh_phan_I(e)` | cộng `e·dt` vào `self.tich_phan` (cộng trước rồi mới nhân), trả về `Ki·∫e dt` |

Thành phần nào có hệ số bằng 0 thì không được gọi, nên xong Bài 1 là chấm được Bài 1.

Thay từng dòng `raise NotImplementedError(...)` bằng code của bạn, chạy ô đó, rồi chạy ô chấm ngay bên dưới (✅ đạt · ❌ sai, kèm gợi ý · ⏳ chưa làm).

In [ ]:
dt = 0.01  # bước thời gian của mô phỏng (giây), giống ../sim/pid.py


class PID:
    def __init__(self, kp, ki, kd):
        self.kp, self.ki, self.kd = kp, ki, kd
        self.tich_phan = 0.0  # ∫e dt cộng dồn từ đầu tới giờ
        self.e_truoc = None   # sai số ở bước trước (None = đây là bước đầu tiên)

    def lenh(self, e):
        """Nhận sai số e = r − y, trả về lệnh tốc độ quay ω = Kp·e + Ki·∫e dt + Kd·de/dt.

        Thành phần nào có hệ số bằng 0 thì bỏ qua, nên Bài 1 chạy được khi chưa làm Bài 2, 3."""
        w = self.thanh_phan_P(e)
        if self.kd:
            w += self.thanh_phan_D(e)
        if self.ki:
            w += self.thanh_phan_I(e)
        return w

    def thanh_phan_P(self, e):
        # Bài 1: lệch bao nhiêu thì bẻ lái bấy nhiêu, theo tỉ lệ Kp.
        raise NotImplementedError("TODO Bài 1: trả về Kp·e")

    def thanh_phan_D(self, e):
        # Bài 2: de/dt ≈ (e − e_trước) / dt. Bước đầu tiên chưa có e_trước thì coi de/dt = 0.
        # Nhớ lưu e vào self.e_truoc để bước sau dùng.
        raise NotImplementedError("TODO Bài 2: trả về Kd·de/dt")

    def thanh_phan_I(self, e):
        # Bài 3: cộng dồn e·dt vào self.tich_phan (cộng TRƯỚC rồi mới nhân), trả về Ki·∫e dt.
        raise NotImplementedError("TODO Bài 3: trả về Ki·∫e dt")


In [ ]:
cham(globals())

## 3. Xem kết quả

In [ ]:
# Chưa xong cả 3 bài thì vẽ bằng lời giải gốc.
try:
    ve(PID, "ket_qua.png")
except NotImplementedError:
    print("Chưa xong cả 3 bài, vẽ bằng lời giải gốc.")
    ve(goc.PID, "ket_qua.png")
display(Image("ket_qua.png"))

## 4. Thử thách

Đổi `kp, ki, kd` rồi chạy lại:

1. Chỉ P, tăng Kp lên 3: lắc nhanh hơn hay chậm hơn?
2. Bỏ D, giữ I (`kd = 0`): chuyện gì xảy ra?
3. Tìm bộ hệ số đưa robot về vạch nhanh nhất mà không vượt quá 10 cm sang phía bên kia.

In [ ]:
kp, ki, kd = 1.5, 0.6, 2.8
Lop = PID
try:
    PID(1, 1, 1).lenh(0.1)
except NotImplementedError:
    Lop = goc.PID  # chưa xong bài thì dùng lời giải gốc
path = goc.chay(Lop(kp, ki, kd))
ys = [p[1] * 100 for p in path]
print(f"Kp={kp} Ki={ki} Kd={kd}: lệch TB 3 s cuối {goc.lech_cuoi(path):+.1f} cm | "
      f"vượt sang phía bên kia tối đa {max(0.0, -min(ys)):.1f} cm | đi được {path[-1][0]:.1f} m")

## Câu hỏi tự kiểm tra

Tự trả lời trước, rồi mở đáp án ở mục *Câu hỏi tự kiểm tra* trong [README của tập](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh/tree/main/tap13).

Code cả series: [github.com/thanhpro82/tu-hoc-robot-tu-hanh](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh)